<a href="https://colab.research.google.com/github/springboardmentor443m-coder/Emotion-Detection-and-Music-Recommendation-System/blob/prabhat/fer_transfer_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
files.upload()


In [ ]:
import os

# kaggle directory
os.makedirs('/root/.kaggle', exist_ok=True)

# Moving kaggle.json
!mv kaggle.json /root/.kaggle/

!chmod 600 /root/.kaggle/kaggle.json


In [ ]:
!kaggle datasets download -d msambare/fer2013


In [ ]:
!unzip fer2013.zip


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Dropout, Flatten, BatchNormalization,GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import MobileNetV2


In [ ]:
import os
import shutil

base_path = "train"

disgust_path = os.path.join(base_path, "disgust")
angry_path = os.path.join(base_path, "angry")

for file in os.listdir(disgust_path):
    shutil.move(
        os.path.join(disgust_path, file),
        os.path.join(angry_path, file)
    )


os.rmdir(disgust_path)





In [ ]:
base_path = "test"

disgust_path = os.path.join(base_path, "disgust")
angry_path = os.path.join(base_path, "angry")

for file in os.listdir(disgust_path):
    shutil.move(
        os.path.join(disgust_path, file),
        os.path.join(angry_path, file)
    )

os.rmdir(disgust_path)




In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Train datagen
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Validation datagen
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

# Train generator
train_generator = train_datagen.flow_from_directory(
    "train",
    target_size=(128,128),
    color_mode="grayscale",
    batch_size=32,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

# Validation generator
val_generator = val_datagen.flow_from_directory(
    "train",
    target_size=(128,128),
    color_mode="grayscale",
    batch_size=32,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)


In [ ]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

from tensorflow.keras.layers import BatchNormalization

# Freeze layers except last 80
for layer in base_model.layers[:-80]:
    layer.trainable = False

for layer in base_model.layers[-80:]:
    layer.trainable = True

# Freeze ALL BatchNorm layers
for layer in base_model.layers:
    if isinstance(layer, BatchNormalization):
        layer.trainable = False


# Input layer
input_layer = Input(shape=(128,128,1))

# Grayscale to RGB conversion
x = Conv2D(3, (1,1), padding='same')(input_layer)

# Pass through MobileNetV2
x = base_model(x)

# Classifier head
x = GlobalAveragePooling2D()(x)

bn = BatchNormalization()
bn.trainable = False
x = bn(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(6, activation='softmax')(x)

# Final model
model = Model(inputs=input_layer, outputs=output)

# Compile
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=30,

)

In [ ]:
test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    "test",
    target_size=(128,128),
    color_mode='grayscale',
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

loss, accuracy = model.evaluate(test_generator)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)